# EXP-027 | MLP 앙상블 (LGB + CAT + XGB + MLP)

GBDT 3모델에 sklearn MLPClassifier를 4번째 모델로 추가. 완전히 다른 학습 방식으로 앙상블 다양성 확보.

| 모델 | 파라미터 |
|---|---|
| LGB | EXP-025 재튜닝 |
| CAT | EXP-011 |
| XGB | EXP-012 |
| MLP | sklearn MLPClassifier, [256→128→64], ReLU+Adam |

| 피처 | FE-v2 + TE 9개 (EXP-023 동일) |
|---|---|
| 기준선 | EXP-023 = 0.74063 |

In [6]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import warnings
from pathlib import Path
from datetime import date

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
from scipy.stats import rankdata

from sklearn.neural_network import MLPClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (roc_auc_score, average_precision_score,
                              f1_score, recall_score, precision_score, accuracy_score)
import lightgbm as lgb
from catboost import CatBoostClassifier, Pool
import xgboost as xgb

from src.preprocessing import preprocess

warnings.filterwarnings('ignore')

DATA_DIR = Path('../data/raw')
OUT_DIR  = Path('../data/submissions')
DOCS_DIR = Path('../docs')
TARGET   = '임신 성공 여부'
SEED     = 42
N_FOLDS  = 5
EXP_NO   = 27
AUTHOR   = '조여진'
CV_STR   = f'Stratified {N_FOLDS}-Fold'

np.random.seed(SEED)

train = pd.read_csv(DATA_DIR / 'train.csv')
test  = pd.read_csv(DATA_DIR / 'test.csv')
sub   = pd.read_csv(DATA_DIR / 'sample_submission.csv')
print(f'train: {train.shape}  /  test: {test.shape}')

train: (256351, 69)  /  test: (90067, 68)


## 1. 피처 준비 (FE v2)

In [7]:
def add_pre_encode_features(df):
    df = df.copy()
    df['기증_난자_여부'] = (df['난자 출처'] == '기증 제공').astype(int)
    df['기증_정자_여부'] = df['정자 출처'].isin(['기증 제공', '배우자 및 기증 제공']).astype(int)
    return df

def add_derived_features(df):
    df = df.copy()
    eps = 1e-6
    df['수정률']    = df['총 생성 배아 수']   / (df['혼합된 난자 수'] + eps)
    df['이식률']    = df['이식된 배아 수']    / (df['총 생성 배아 수'] + eps)
    df['저장률']    = df['저장된 배아 수']    / (df['총 생성 배아 수'] + eps)
    df['ICSI_비율'] = df['미세주입된 난자 수'] / (df['혼합된 난자 수'] + eps)
    df['배아_발달일']     = df['배아 이식 경과일'] - df['난자 혼합 경과일']
    df['신선_시술_여부']  = df['수집된 신선 난자 수'].notna().astype(int)
    male_cols   = ['남성 주 불임 원인','남성 부 불임 원인','불임 원인 - 남성 요인']
    female_cols = ['여성 주 불임 원인','여성 부 불임 원인','불임 원인 - 난관 질환',
                   '불임 원인 - 배란 장애','불임 원인 - 자궁내막증','불임 원인 - 자궁경부 문제']
    couple_cols = ['부부 주 불임 원인','부부 부 불임 원인']
    sperm_cols  = ['불임 원인 - 정자 농도','불임 원인 - 정자 운동성',
                   '불임 원인 - 정자 형태','불임 원인 - 정자 면역학적 요인']
    all_cause   = male_cols + female_cols + couple_cols + sperm_cols + ['불명확 불임 원인']
    df['남성_불임_합계']       = df[male_cols].sum(axis=1)
    df['여성_불임_합계']       = df[female_cols].sum(axis=1)
    df['부부_불임_합계']       = df[couple_cols].sum(axis=1)
    df['정자_문제_합계']       = df[sperm_cols].sum(axis=1)
    df['총_불임원인_수']       = df[all_cause].sum(axis=1)
    df['임신시도기록_있음']     = df['임신 시도 또는 마지막 임신 경과 연수'].notna().astype(int)
    df['신선_난자_저장_있음']   = (df['저장된 신선 난자 수'] > 0).astype(int)
    df['나이_시술횟수_상호작용'] = df['시술 당시 나이'] * df['총 시술 횟수']
    df['임신_성공률']   = df['총 임신 횟수'] / (df['총 시술 횟수'] + eps)
    df['시술_실패_횟수'] = (df['총 시술 횟수'] - df['총 임신 횟수']).clip(lower=0)
    egg_count = df['수집된 신선 난자 수'].fillna(-1)
    df['최적_난자수']    = ((egg_count >= 5) & (egg_count <= 15)).astype(int)
    df['나이_이식배아수'] = df['시술 당시 나이'] * df['이식된 배아 수'].fillna(0)
    return df

train_fe = add_pre_encode_features(train)
test_fe  = add_pre_encode_features(test)
X_train, X_test = preprocess(train_fe, test_fe)
X_train = add_derived_features(X_train)
X_test  = add_derived_features(X_test)
y_train = train[TARGET]
neg_pos_ratio = (y_train == 0).sum() / (y_train == 1).sum()
print(f'X_train: {X_train.shape}  /  X_test: {X_test.shape}')

X_train: (256351, 89)  /  X_test: (90067, 89)


## 2. 모델 설정

In [8]:
TE_COLS  = [
    '시술 시기 코드', '시술 유형', '특정 시술 유형',
    '배란 유도 유형', '난자 출처', '정자 출처',
    '시술 당시 나이', '총 시술 횟수', '총 임신 횟수',
]
TE_COLS  = [c for c in TE_COLS if c in X_train.columns]
TE_ALPHA = 10

LGB_PARAMS = dict(
    objective='binary', metric='auc', verbosity=-1, seed=SEED,
    num_threads=-1, is_unbalance=True, bagging_freq=1,
    learning_rate=0.01369375634677629, num_leaves=126, max_depth=5,
    min_child_samples=182, feature_fraction=0.3284851940428519,
    bagging_fraction=0.6459035591898032, lambda_l1=1.2016221908502045,
    lambda_l2=0.02431385472202543, min_gain_to_split=0.3216337223328029,
)
CAT_PARAMS = dict(
    iterations=2000, loss_function='Logloss', eval_metric='AUC',
    auto_class_weights='Balanced', random_seed=SEED,
    thread_count=-1, verbose=False, early_stopping_rounds=100,
    learning_rate=0.018758723768855998, depth=6,
    l2_leaf_reg=9.189608434163782, min_data_in_leaf=19,
    subsample=0.8170921295501483, colsample_bylevel=0.6936810336930781,
)
XGB_PARAMS = dict(
    n_estimators=2000, scale_pos_weight=neg_pos_ratio,
    eval_metric='auc', tree_method='hist', random_state=SEED,
    n_jobs=-1, verbosity=0, early_stopping_rounds=100,
    learning_rate=0.05520069867907647, max_depth=4, min_child_weight=59,
    subsample=0.7663066457187595, colsample_bytree=0.6581836436885355,
    reg_alpha=8.692038126211928, reg_lambda=0.23932562420374562,
)

MLP_CFG = dict(
    hidden_layer_sizes=(256, 128, 64),
    activation='relu',
    solver='adam',
    alpha=1e-5,
    batch_size=2048,
    learning_rate_init=1e-3,
    max_iter=100,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=10,
    verbose=False,
)
print('설정 완료')
print(f'MLP 구조: Input → {" → ".join(str(d) for d in MLP_CFG["hidden_layer_sizes"])} → 1')

설정 완료
MLP 구조: Input → 256 → 128 → 64 → 1


## 3. MLP 클래스 & 학습 함수

In [9]:
def train_mlp(X_tr_s, y_tr_arr, X_val_s, y_val_arr, cfg, seed):
    # MLPClassifier는 sample_weight 미지원 → minority class 랜덤 오버샘플링
    rng     = np.random.default_rng(seed)
    pos_idx = np.where(y_tr_arr == 1)[0]
    neg_idx = np.where(y_tr_arr == 0)[0]
    extra   = rng.choice(pos_idx, size=len(neg_idx) - len(pos_idx), replace=True)
    idx     = np.concatenate([np.arange(len(y_tr_arr)), extra])
    X_bal   = X_tr_s[idx]
    y_bal   = y_tr_arr[idx]

    model = MLPClassifier(**cfg, random_state=seed)
    model.fit(X_bal, y_bal)
    val_pred = model.predict_proba(X_val_s)[:, 1]
    best_auc = roc_auc_score(y_val_arr, val_pred)
    return model, best_auc


def predict_mlp(model, X_s):
    return model.predict_proba(X_s)[:, 1]


print('MLP 함수 정의 완료 (sklearn MLPClassifier + 오버샘플링)')

MLP 함수 정의 완료 (sklearn MLPClassifier + 오버샘플링)


## 4. 학습 (4-model KFold)

In [ ]:
import time

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

oof_lgb = np.zeros(len(X_train)); test_lgb = np.zeros(len(X_test))
oof_cat = np.zeros(len(X_train)); test_cat = np.zeros(len(X_test))
oof_xgb = np.zeros(len(X_train)); test_xgb = np.zeros(len(X_test))
oof_mlp = np.zeros(len(X_train)); test_mlp = np.zeros(len(X_test))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train), 1):
    print(f'\n── Fold {fold}/{N_FOLDS} ──────────────────────────────')
    X_tr  = X_train.iloc[tr_idx].copy()
    X_val = X_train.iloc[val_idx].copy()
    y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]
    X_te  = X_test.copy()
    global_mean = y_tr.mean()

    # ── KFold Target Encoding ──────────────────────────────────────
    for col in TE_COLS:
        te_col   = f'{col}_te'
        grp      = y_tr.groupby(X_tr[col])
        cnt      = grp.count()
        mean_t   = grp.mean()
        smoothed = (cnt * mean_t + TE_ALPHA * global_mean) / (cnt + TE_ALPHA)
        X_tr[te_col]  = X_tr[col].map(smoothed).fillna(global_mean)
        X_val[te_col] = X_val[col].map(smoothed).fillna(global_mean)
        X_te[te_col]  = X_te[col].map(smoothed).fillna(global_mean)
    # ──────────────────────────────────────────────────────────────

    # LGB
    t0 = time.time()
    ds_tr      = lgb.Dataset(X_tr, label=y_tr)
    ds_val_lgb = lgb.Dataset(X_val, label=y_val, reference=ds_tr)
    m = lgb.train(LGB_PARAMS, ds_tr, num_boost_round=3000, valid_sets=[ds_val_lgb],
                  callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(-1)])
    oof_lgb[val_idx] = m.predict(X_val); test_lgb += m.predict(X_te) / N_FOLDS
    print(f'  LGB  AUC={roc_auc_score(y_val, oof_lgb[val_idx]):.5f}  ({time.time()-t0:.0f}s)')

    # CAT
    t0 = time.time()
    m = CatBoostClassifier(**CAT_PARAMS)
    m.fit(X_tr, y_tr, eval_set=Pool(X_val, y_val), use_best_model=True)
    oof_cat[val_idx] = m.predict_proba(X_val)[:, 1]; test_cat += m.predict_proba(X_te)[:, 1] / N_FOLDS
    print(f'  CAT  AUC={roc_auc_score(y_val, oof_cat[val_idx]):.5f}  ({time.time()-t0:.0f}s)')

    # XGB
    t0 = time.time()
    m = xgb.XGBClassifier(**XGB_PARAMS)
    m.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    oof_xgb[val_idx] = m.predict_proba(X_val)[:, 1]; test_xgb += m.predict_proba(X_te)[:, 1] / N_FOLDS
    print(f'  XGB  AUC={roc_auc_score(y_val, oof_xgb[val_idx]):.5f}  ({time.time()-t0:.0f}s)')

    # MLP — NaN 처리 + StandardScaler (train fold 기준)
    t0 = time.time()
    medians  = X_tr.median()
    X_tr_np  = X_tr.fillna(medians).values.astype(np.float32)
    X_val_np = X_val.fillna(medians).values.astype(np.float32)
    X_te_np  = X_te.fillna(medians).values.astype(np.float32)
    scaler   = StandardScaler()
    X_tr_s   = scaler.fit_transform(X_tr_np)
    X_val_s  = scaler.transform(X_val_np)
    X_te_s   = scaler.transform(X_te_np)
    mlp_model, mlp_best = train_mlp(
        X_tr_s, y_tr.values, X_val_s, y_val.values,
        cfg=MLP_CFG, seed=SEED + fold
    )
    oof_mlp[val_idx] = predict_mlp(mlp_model, X_val_s)
    test_mlp += predict_mlp(mlp_model, X_te_s) / N_FOLDS
    print(f'  MLP  AUC={roc_auc_score(y_val, oof_mlp[val_idx]):.5f}  best={mlp_best:.5f}  ({time.time()-t0:.0f}s)')

auc_lgb = roc_auc_score(y_train, oof_lgb)
auc_cat = roc_auc_score(y_train, oof_cat)
auc_xgb = roc_auc_score(y_train, oof_xgb)
auc_mlp = roc_auc_score(y_train, oof_mlp)
n_features_final = X_tr.shape[1]
print(f'\nOOF  LGB={auc_lgb:.5f}  CAT={auc_cat:.5f}  XGB={auc_xgb:.5f}  MLP={auc_mlp:.5f}')


── Fold 1/5 ──────────────────────────────
  LGB  AUC=0.73847  (20s)
  CAT  AUC=0.73827  (29s)
  XGB  AUC=0.73813  (24s)
  MLP  AUC=0.68428  best=0.68428  (721s)

── Fold 2/5 ──────────────────────────────
  LGB  AUC=0.74355  (27s)
  CAT  AUC=0.74338  (55s)
  XGB  AUC=0.74316  (26s)
  MLP  AUC=0.67578  best=0.67578  (978s)

── Fold 3/5 ──────────────────────────────
  LGB  AUC=0.74053  (20s)
  CAT  AUC=0.74020  (35s)
  XGB  AUC=0.74015  (25s)
  MLP  AUC=0.67559  best=0.67559  (510s)

── Fold 4/5 ──────────────────────────────
  LGB  AUC=0.73884  (19s)
  CAT  AUC=0.73864  (34s)
  XGB  AUC=0.73882  (24s)
  MLP  AUC=0.67073  best=0.67073  (562s)

── Fold 5/5 ──────────────────────────────
  LGB  AUC=0.74102  (22s)
  CAT  AUC=0.74062  (47s)
  XGB  AUC=0.74154  (30s)


## 5. 앙상블 & 결과 비교

In [14]:
oofs  = np.stack([oof_lgb, oof_cat, oof_xgb, oof_mlp], axis=1)
tests = np.stack([test_lgb, test_cat, test_xgb, test_mlp], axis=1)
aucs  = np.array([auc_lgb, auc_cat, auc_xgb, auc_mlp])

results = {}
results['Simple Average'] = (roc_auc_score(y_train, oofs.mean(axis=1)),
                              oofs.mean(axis=1), tests.mean(axis=1))

w_auc = aucs / aucs.sum()
results['AUC-weighted'] = (roc_auc_score(y_train, (oofs * w_auc).sum(axis=1)),
                            (oofs * w_auc).sum(axis=1), (tests * w_auc).sum(axis=1))

oof_ranks  = np.stack([rankdata(oofs[:, i]) for i in range(4)], axis=1)
test_ranks = np.stack([rankdata(tests[:, i]) for i in range(4)], axis=1)
results['Rank Average'] = (roc_auc_score(y_train, oof_ranks.mean(axis=1)),
                            oof_ranks.mean(axis=1), test_ranks.mean(axis=1))

def optuna_obj(trial):
    w = np.array([trial.suggest_float('w_lgb', 0, 1),
                  trial.suggest_float('w_cat', 0, 1),
                  trial.suggest_float('w_xgb', 0, 1),
                  trial.suggest_float('w_mlp', 0, 1)])
    w /= w.sum()
    return roc_auc_score(y_train, (oofs * w).sum(axis=1))

study = optuna.create_study(direction='maximize',
                             sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(optuna_obj, n_trials=300, show_progress_bar=False)
best_w = np.array([study.best_params['w_lgb'], study.best_params['w_cat'],
                   study.best_params['w_xgb'], study.best_params['w_mlp']])
best_w /= best_w.sum()
results['Optuna Weights'] = (roc_auc_score(y_train, (oofs * best_w).sum(axis=1)),
                              (oofs * best_w).sum(axis=1), (tests * best_w).sum(axis=1))

BASELINE = 0.74063
print('=' * 70)
print(f'  개별: LGB={auc_lgb:.5f}  CAT={auc_cat:.5f}  XGB={auc_xgb:.5f}  MLP={auc_mlp:.5f}')
print(f'  EXP-023 기준선: {BASELINE}')
print('-' * 70)
best_method, best_auc, best_oof_pred, best_test = '', 0, None, None
for method, (auc, oof_pred, test_pred) in results.items():
    flag = ' ←' if auc > max(aucs) else ''
    print(f'  {method:20s}: {auc:.5f}  ({auc - BASELINE:+.5f} vs EXP-023){flag}')
    if auc > best_auc:
        best_auc, best_method, best_oof_pred, best_test = auc, method, oof_pred, test_pred
print('=' * 70)
print(f'\n최적: {best_method}  AUC={best_auc:.5f}')
print(f'Optuna 가중치  LGB={best_w[0]:.3f}  CAT={best_w[1]:.3f}  '
      f'XGB={best_w[2]:.3f}  MLP={best_w[3]:.3f}')

  개별: LGB=0.74047  CAT=0.74022  XGB=0.74035  MLP=0.67588
  EXP-023 기준선: 0.74063
----------------------------------------------------------------------
  Simple Average      : 0.72792  (-0.01271 vs EXP-023)
  AUC-weighted        : 0.72933  (-0.01130 vs EXP-023)
  Rank Average        : 0.73694  (-0.00369 vs EXP-023)
  Optuna Weights      : 0.74063  (+0.00000 vs EXP-023) ←

최적: Optuna Weights  AUC=0.74063
Optuna 가중치  LGB=0.405  CAT=0.275  XGB=0.314  MLP=0.006


## 6. Submission 저장 & 실험 기록

In [15]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
sub['probability'] = best_test
auc_str   = f'{best_auc:.4f}'.replace('.', '')
out_fname = f'submission_exp{EXP_NO:03d}_{AUTHOR}_{auc_str}.csv'
sub.to_csv(OUT_DIR / out_fname, index=False)
print(f'저장: {OUT_DIR / out_fname}')

저장: ../data/submissions/submission_exp027_조여진_07406.csv


In [16]:
from openpyxl import load_workbook
from openpyxl.styles import Font, Alignment, Border, Side, PatternFill

def log_to_leaderboard(exp_no, author, model_name, params_str,
                        f1, recall, precision, accuracy, oof_auc,
                        cv_strategy, preprocessing_ver, n_features,
                        imbalance_method, submitted, hackathon_score,
                        file_name, notes='', insights=''):
    lb_path = DOCS_DIR / 'leaderboard.xlsx'
    wb = load_workbook(lb_path)
    ws = wb['리더보드']
    exp_label = f'EXP-{exp_no:03d}'
    next_row = ws.max_row + 1
    for r in range(2, ws.max_row + 1):
        val = ws.cell(row=r, column=2).value
        if val == exp_label:
            next_row = r; break
        if ws.cell(row=r, column=1).value is None or str(ws.cell(row=r, column=1).value).strip() == '':
            next_row = r; break
    values = [str(date.today()), exp_label, author, model_name, params_str,
              round(f1,5), round(recall,5), round(precision,5), round(accuracy,5), round(oof_auc,5),
              cv_strategy, preprocessing_ver, n_features, imbalance_method,
              submitted, hackathon_score, file_name, notes, insights]
    thin = Side(style='thin', color='B0B8D0')
    border = Border(left=thin, right=thin, top=thin, bottom=thin)
    fill = PatternFill('solid', fgColor='EEF2FA') if next_row % 2 == 0 else None
    font = Font(name='맑은 고딕', size=10)
    center = Alignment(horizontal='center', vertical='center', wrap_text=True)
    left   = Alignment(horizontal='left',   vertical='center', wrap_text=True)
    left_cols = {4, 5, 12, 14, 17, 18, 19}
    for c_idx, val in enumerate(values, start=1):
        cell = ws.cell(row=next_row, column=c_idx, value=val)
        cell.font = font; cell.border = border
        cell.alignment = left if c_idx in left_cols else center
        if fill: cell.fill = fill
        if c_idx in range(6, 11) or c_idx == 16: cell.number_format = '0.00000'
    wb.save(lb_path)
    print(f'[leaderboard.xlsx] EXP-{exp_no:03d} 기록 완료 (row {next_row})')

oof_binary = (best_oof_pred >= 0.5).astype(int)
arch_str   = '-'.join(str(d) for d in MLP_CFG['hidden_layer_sizes'])
params_str = (f'FE-v2+TE | LGB(EXP-025)+CAT(EXP-011)+XGB(EXP-012)+MLP([{arch_str}]) | '
              f'best={best_method}')
NOTES    = (f'MLP(sklearn): [{arch_str}] ReLU+Adam '
            f'lr={MLP_CFG["learning_rate_init"]} bs={MLP_CFG["batch_size"]} '
            f'alpha={MLP_CFG["alpha"]}')
INSIGHTS = (f'EXP-023 대비 {best_auc - BASELINE:+.5f} | '
            f'MLP OOF={auc_mlp:.5f} | Optuna MLP weight={best_w[3]:.3f} | '
            f'피처 수 {n_features_final}')

log_to_leaderboard(
    EXP_NO, AUTHOR, 'Ensemble(LGB+CAT+XGB+MLP)', params_str,
    f1_score(y_train, oof_binary), recall_score(y_train, oof_binary),
    precision_score(y_train, oof_binary), accuracy_score(y_train, oof_binary),
    best_auc, CV_STR, 'v2+TE', n_features_final,
    'is_unbalance / Balanced / scale_pos_weight / oversample(balanced)',
    'N', None, 'notebooks/23_mlp_ensemble_yjcho.ipynb', NOTES, INSIGHTS
)

[leaderboard.xlsx] EXP-027 기록 완료 (row 25)
